In [1]:
import pandas as pd

In [2]:
! pip install pyarrow
! pip install fastparquet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pyarrow.parquet as pq
import pyarrow as pa

table = pq.read_table('../compria-reactions/reactions.parquet')
table = table.cast(
    pa.schema([(f.name, pa.string() if pa.types.is_large_string(f.type) else f.type) 
               for f in table.schema])
)
df_reactions = table.to_pandas()

table = pq.read_table('../compria-votes/votes.parquet')
table = table.cast(
    pa.schema([(f.name, pa.string() if pa.types.is_large_string(f.type) else f.type) 
               for f in table.schema])
)
df_votes = table.to_pandas()

table = pq.read_table('../compria-conversations/conversations.parquet')
table = table.cast(
    pa.schema([(f.name, pa.string() if pa.types.is_large_string(f.type) else f.type) 
               for f in table.schema])
)
df_conversations = table.to_pandas()



In [4]:
df_reactions.iloc[0]

id                                                                      202099
timestamp                                           2025-04-23 19:32:46.687338
model_a_name                                                       gemma-3-12b
model_b_name                                                         command-a
refers_to_model                                                      command-a
msg_index                                                                    1
opening_msg                  Quelle est la stratégie commerciale du champag...
conversation_a               [{'content': 'Quelle est la stratégie commerci...
conversation_b               [{'content': 'Quelle est la stratégie commerci...
model_pos                                                                    b
conv_turns                                                                   1
conversation_pair_id         1a62544c50474085b25240991e9fc620-6e6ffca415f44...
conv_a_id                                     1a6254

In [5]:
df_votes.iloc[0]

id                                                                             112580
timestamp                                                  2025-10-28 17:02:44.266579
model_a_name                                                         gemini-2.5-flash
model_b_name                                                              grok-4-fast
model_pair_name                                       [gemini-2.5-flash, grok-4-fast]
chosen_model_name                                                                 NaN
opening_msg                         crée des cartes types dixit sur le thème des m...
both_equal                                                                       True
conversation_a                      [{'content': 'crée des cartes types dixit sur ...
conversation_b                      [{'content': 'crée des cartes types dixit sur ...
conv_turns                                                                          2
selected_category                                     

In [6]:
df_conversations.iloc[0]

id                                                                      1562219
timestamp                                            2026-03-06 19:37:04.399632
model_a_name                                             gemini-3.1-pro-preview
model_b_name                                                  claude-4-6-sonnet
conversation_a                [{'content': 'Comment protéger les métiers art...
conversation_b                [{'content': 'Comment protéger les métiers art...
conv_turns                                                                    1
conversation_pair_id          14edd982fc504dbe9835702fa2127ea5-1b4887d008ba4...
conv_a_id                                      14edd982fc504dbe9835702fa2127ea5
conv_b_id                                      1b4887d008ba47ac8ecba23600dcb4cf
session_hash                               934e7178-476f-406d-a0e7-acdfd8d3e24b
visitor_id                                     5ddc8517e4630de952fa033d8a54f73a
model_pair_name                      {cl

In [35]:
df_reactions_filtered = df_reactions.groupby(['conversation_pair_id','msg_index']).filter(lambda x: len(x) >= 2)
filtered_groups = df_reactions_filtered.groupby(['conversation_pair_id','msg_index'])

In [36]:
def taux_accord(x):
    vc = x.value_counts(dropna=False)  # dropna=False counts None/NaN too
    return vc.iloc[0] / len(x)

taux_accords_creative = filtered_groups['creative'].agg(taux_accord)

In [38]:
taux_accords_creative.mean()

np.float64(0.9527999755993412)